### Quét các thiết bị Bluetooth

In [14]:
import asyncio
from bleak import BleakScanner


async def main():
    print("Scanning for BLE devices...\n")

    devices = await BleakScanner.discover(timeout=10)

    for device in devices:
        print(device)


await main()

Scanning for BLE devices...

DD668754-AEA8-FA6C-F522-EDFD1C603618: 75" QLED
C62ABE88-8292-E5F8-3CCD-AEF19CC5F95D: None
67207AEC-5CCD-1E7F-C3DB-A7D567385193: [TV] UA48J5500
3AA6C6FE-BCB8-54FC-7D29-194243B35024: Google Pixel 9
B8775202-8A4C-D760-E4AC-673EE42B1B56: None
62780A7A-0F91-CA11-5534-7541D13CA933: Polar H10 13EB8632
FD7C743E-AE3D-88A0-D9AC-3496F59004A0: None
7C2E6A41-1C3D-9628-E11D-3D0949C19423: None
BE00377B-9B57-753A-935B-2CB912DF1EC2: None
EDB304B5-459E-8373-0D8B-0C44C512D8EB: None
1DA787A9-C98C-90F9-F874-B13DCB12FA49: net
B21B0786-9B44-B73F-23CC-E120A5977C37: None
C8E4585D-AA01-40EC-8EC1-108065759423: None
ED9920AE-2623-C056-6D9F-BF104D8D9DEE: None
249F50C0-79ED-4668-3725-5F91CE04C29F: None
F614CC6C-6D9B-41EC-C6DE-144EBCBA2D2A: None
B664942D-7010-E464-7A99-58E7C951521D: None
A9045DBC-0A7A-CFF4-BF97-DD9FAFA3FE9C: None


### Kết nối với IMU và thu vào raw_result (100Hz)

In [15]:
import asyncio
from bleak import BleakClient

H10_ADDRESS = "62780A7A-0F91-CA11-5534-7541D13CA933"

PMD_CP_UUID = "FB005C81-02E7-F387-1CAD-8ACD2D8DF0C8"
PMD_DATA_UUID = "FB005C82-02E7-F387-1CAD-8ACD2D8DF0C8"

DURATION_SECONDS = 10

# Lưu toàn bộ packet RAW
raw_result = []


def control_handler(sender, data):
    print("PMD CONTROL:", data.hex(" "))


def data_handler(sender, data):
    # Không decode ở đây
    raw_result.append(bytes(data))


async def collect_raw():

    raw_result.clear()

    async with BleakClient(
        H10_ADDRESS,
        timeout=15
    ) as client:

        print("Connected:", client.is_connected)

        # Enable notifications
        await client.start_notify(
            PMD_CP_UUID,
            control_handler
        )

        await client.start_notify(
            PMD_DATA_UUID,
            data_handler
        )

        print("PMD notifications enabled.")

        # -----------------------------------------
        # ACC 200 Hz / 16-bit / 8G
        # -----------------------------------------

        ACC_START = bytes([
            0x02,
            0x02,

            0x00, 0x01, 0x64,        # 100 Hz
            0x00, 0x01, 0x01, 0x10,  # 16-bit
            0x00, 0x02, 0x01, 0x08,  # 8G
            0x00
        ])

        ACC_STOP = bytes([
            0x03,
            0x02
        ])

        print("Starting ACC...")
        print("Command:", ACC_START.hex(" "))

        await client.write_gatt_char(
            PMD_CP_UUID,
            ACC_START,
            response=True
        )

        print("Collecting RAW data...")

        await asyncio.sleep(DURATION_SECONDS)

        print("Stopping ACC...")

        try:
            await client.write_gatt_char(
                PMD_CP_UUID,
                ACC_STOP,
                response=True
            )
        except Exception as e:
            print("STOP error:", e)

        await client.stop_notify(PMD_DATA_UUID)
        await client.stop_notify(PMD_CP_UUID)

    print("\nFinished.")
    print("Number of RAW packets:", len(raw_result))


await collect_raw()

Connected: True
PMD notifications enabled.
Starting ACC...
Command: 02 02 00 01 64 00 01 01 10 00 02 01 08 00
PMD CONTROL: f0 02 02 00 00 01
Stopping ACC...
PMD CONTROL: f0 03 02 00 00

Finished.
Number of RAW packets: 27


In [17]:
import struct
import pandas as pd
import numpy as np


# ============================================================
# CONFIG
# ============================================================

SAMPLE_RATE = 100.0
ACC_RANGE_G = 8.0
ACC_SCALE = ACC_RANGE_G / 32768.0

decoded_result = []


# ============================================================
# DECODE 1 PACKET
# ============================================================

def decode_acc_packet(packet):

    if len(packet) < 16:
        return []

    # ACC
    if packet[0] != 0x02:
        return []

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    timestamp_ns = int.from_bytes(
        packet[1:9],
        byteorder="little",
        signed=False
    )

    timestamp_s = timestamp_ns / 1e9

    # Byte 9
    frame_type = packet[9]

    # --------------------------------------------------------
    # ACC DATA
    #
    # Packet 226 bytes:
    #
    # 10-byte header
    # + 36 samples × 6 bytes
    #
    # 10 + 36*6 = 226
    # --------------------------------------------------------

    offset = 10

    samples = []

    sample_index = 0

    while offset + 6 <= len(packet):

        raw_x, raw_y, raw_z = struct.unpack_from(
            "<hhh",
            packet,
            offset
        )

        acc_x = raw_x * ACC_SCALE
        acc_y = raw_y * ACC_SCALE
        acc_z = raw_z * ACC_SCALE

        sample_timestamp = (
            timestamp_s +
            sample_index / SAMPLE_RATE
        )

        samples.append({
            "TimeStamp(s)": sample_timestamp,
            "AccX": acc_x,
            "AccY": acc_y,
            "AccZ": acc_z
        })

        offset += 6
        sample_index += 1

    return samples


# ============================================================
# DECODE ALL RAW PACKETS
# ============================================================

decoded_result.clear()

packet_info = []

for packet_index, packet in enumerate(raw_result):

    samples = decode_acc_packet(packet)

    decoded_result.extend(samples)

    packet_info.append({
        "Packet": packet_index,
        "Length": len(packet),
        "Samples": len(samples)
    })


# ============================================================
# DATAFRAME
# ============================================================

df_acc = pd.DataFrame(decoded_result)


# ============================================================
# BASIC INFORMATION
# ============================================================

print("=" * 60)
print("RAW DATA")
print("=" * 60)

print("Số RAW packets :", len(raw_result))

if raw_result:
    print("Packet đầu tiên :", len(raw_result[0]), "bytes")


print("\n" + "=" * 60)
print("DECODED DATA")
print("=" * 60)

print("Số samples :", len(df_acc))


# ============================================================
# PACKET INFORMATION
# ============================================================

df_packet_info = pd.DataFrame(packet_info)

print("\n" + "=" * 60)
print("PACKET INFORMATION")
print("=" * 60)

print(df_packet_info.head(10))

print("\nSamples / packet:")
print(
    df_packet_info["Samples"]
    .value_counts()
    .sort_index()
)


# ============================================================
# FIRST SAMPLES
# ============================================================

print("\n" + "=" * 60)
print("20 SAMPLES ĐẦU")
print("=" * 60)

print(
    df_acc.head(20).to_string(index=False)
)


# ============================================================
# ACC STATISTICS
# ============================================================

print("\n" + "=" * 60)
print("ACC STATISTICS")
print("=" * 60)

print(
    df_acc[["AccX", "AccY", "AccZ"]].describe()
)


# ============================================================
# ACC MAGNITUDE
# ============================================================

df_acc["AccMagnitude"] = np.sqrt(
    df_acc["AccX"] ** 2 +
    df_acc["AccY"] ** 2 +
    df_acc["AccZ"] ** 2
)


print("\n" + "=" * 60)
print("ACC MAGNITUDE")
print("=" * 60)

print(
    df_acc["AccMagnitude"].describe()
)


# ============================================================
# TIMESTAMP / SAMPLE RATE
# ============================================================

if len(df_acc) > 1:

    dt = df_acc["TimeStamp(s)"].diff().dropna()

    print("\n" + "=" * 60)
    print("TIMESTAMP / SAMPLE RATE")
    print("=" * 60)

    print("dt mean     :", dt.mean(), "s")
    print("Sample rate :", 1 / dt.mean(), "Hz")

    print("\nFirst 20 dt:")
    print(
        dt.head(20).to_string(index=False)
    )


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)

print(
    "NaN:",
    df_acc[["AccX", "AccY", "AccZ"]]
    .isna()
    .sum()
    .sum()
)

print(
    "AccX range:",
    df_acc["AccX"].min(),
    "→",
    df_acc["AccX"].max()
)

print(
    "AccY range:",
    df_acc["AccY"].min(),
    "→",
    df_acc["AccY"].max()
)

print(
    "AccZ range:",
    df_acc["AccZ"].min(),
    "→",
    df_acc["AccZ"].max()
)

print(
    "Magnitude mean:",
    df_acc["AccMagnitude"].mean(),
    "G"
)

print(
    "Magnitude median:",
    df_acc["AccMagnitude"].median(),
    "G"
)

print(
    "Magnitude max:",
    df_acc["AccMagnitude"].max(),
    "G"
)

RAW DATA
Số RAW packets : 27
Packet đầu tiên : 226 bytes

DECODED DATA
Số samples : 972

PACKET INFORMATION
   Packet  Length  Samples
0       0     226       36
1       1     226       36
2       2     226       36
3       3     226       36
4       4     226       36
5       5     226       36
6       6     226       36
7       7     226       36
8       8     226       36
9       9     226       36

Samples / packet:
Samples
36    27
Name: count, dtype: int64

20 SAMPLES ĐẦU
 TimeStamp(s)      AccX      AccY     AccZ
 5.996168e+08 -0.101074 -0.057617 0.222412
 5.996168e+08 -0.101807 -0.058350 0.218506
 5.996168e+08 -0.101562 -0.058838 0.220215
 5.996168e+08 -0.100830 -0.059814 0.223145
 5.996168e+08 -0.100586 -0.060059 0.224121
 5.996168e+08 -0.101562 -0.059814 0.222900
 5.996168e+08 -0.100098 -0.059814 0.223877
 5.996168e+08 -0.098877 -0.058350 0.216797
 5.996168e+08 -0.102295 -0.057129 0.213867
 5.996168e+08 -0.101074 -0.058105 0.217773
 5.996168e+08 -0.100098 -0.059570 0.218750
 